# Basic Decision Tree from Scratch - Classification (Gini) - Optimization

In this notebook, we are going to optimize the Gini search algorithm we had. The optimization can be organized into:

- Optimization 1 : Sliding Windows
- Optimization 2 : Cumulative Sum
- Optimization 3 : Vectorization
- Optimization 4 : Reduce boundaries and remove feature values that are duplicates

## Dataset

Dataset: This is a makeup dataset that describe how much study hours a student put in and whether if they will either pass or fail their exam

| Study Hours  | Pass Exam |
| ----- | ----- |
| 1    | No    |
| 2    | No    |
| 3    | No    |
| 4    | No    |
| 5    | Yes   |
| 6    | Yes   |
| 7    | Yes   |
| 8    | Yes   |
| 9    | Yes   |
| 10    | Yes   |
| 11    | Yes   |

In [1]:
import pandas as pd
import numpy as np
import json
from typing import Optional

In [2]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [3]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


## Gini Formula

### Gini Impurity Formula

For a node:

$$\text{Gini} = 1 - \sum_{k=1}^{K} p_k^2$$


Where:

- K = number of classes
- $p_k$ = proportion of class (k) in the node


#### Binary classification Example (0/1 case)

$$\text{Gini} = 1 - (p_0^2 + p_1^2)$$

Where:

- $p_0 = \frac{0}{N}$
- $p_1 = \frac{1}{N}$

**Intuition**

| Case           | Gini                          |
| -------------- | ----------------------------- |
| all same class | 0 (pure)                      |
| 50/50 split    | 0.5 (max impurity for binary) |

---

### Weighted Gini (for a split)

When you split a node into LEFT and RIGHT:

$$ \text{Weighted Gini} = \frac{N_L}{N} \cdot Gini(L) + \frac{N_R}{N} \cdot Gini(R)$$

Where:

- $N_L$ = number of samples in left node
- $N_R$ = number of samples in right node
- $N = N_L + N_R$


**Intuition**

Weighted Gini means:

> "how impure are the children, adjusted by their size?"

So:

- big bad child → matters a lot
- small bad child → matters less

### Gini split procedure:
- Sort feature values
- For each adjacent pair:
    - compute midpoint threshold
- For each threshold:
    - split data
    - compute weighted Gini
- Choose best threshold

## Midpoint Optimization

Midpoint can also be optimized but this is low priority as we optimized further, we only computed selected midpoint instead of computing all midpoint.

In [4]:
# compute mid points between values of studied
mid_points = []
for i in range(len(df['studied']) - 1):
    mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
    # [i] refer to the fist value and [i+1] refer to the second value and so on
    mid_points.append(float(mid_point))

print(mid_points)

[1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 11.5]


In [5]:
x = df['studied'].values
x

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [6]:
print(x[:-1])
print(x[1:])

[ 1  2  3  4  5  6  7  8  9 10 11]
[ 2  3  4  5  6  7  8  9 10 11 12]


In [7]:
mid_points = (x[:-1] + x[1:]) / 2
mid_points

array([ 1.5,  2.5,  3.5,  4.5,  5.5,  6.5,  7.5,  8.5,  9.5, 10.5, 11.5])

## Current Gini Algorithm

The current gini search algorithm we use `for loop` to calculate all midpoint and weighted gini. Then we select the midpoint with least weighted gini.

The current loop is resource intensive because for every midpoint, we split the features into 2 portion.

In [8]:
def gini_v1(y: pd.Series)-> float:
    '''
    Compute the Gini impurity for a set of labels.
    y: array-like of shape (n_samples,) - target labels for the samples in the node
    return: float - Gini impurity of the node
    '''
    if len(y) == 0:
        return 0
    p0 = (y == 0).sum() / len(y)
    p1 = (y == 1).sum() / len(y)

    gini = 1 - (p0**2 + p1**2)
    return gini

In [9]:
# weighted gini impurity
def weighted_gini_v1(df_left: pd.DataFrame, df_right: pd.DataFrame) -> float:
    total_samples = len(df_left) + len(df_right)
    gini_left = gini_v1(df_left['passed'])
    gini_right = gini_v1(df_right['passed'])
    
    weighted_gini = (len(df_left) / total_samples) * gini_left + (len(df_right) / total_samples) * gini_right
    return weighted_gini

In [10]:
def best_gini_split(df: pd.DataFrame) -> tuple[Optional[float], float]:
    best_gini = float('inf')
    best_split_value = None
    df_sorted = df.sort_values(by='studied').reset_index(drop=True)
    
    for i in range(len(df_sorted['studied']) - 1):
        mid_point = (df_sorted['studied'][i] + df_sorted['studied'][i+1]) / 2
        df_left = df_sorted[df_sorted['studied'] <= mid_point]
        df_right = df_sorted[df_sorted['studied'] > mid_point]
        
        current_gini = weighted_gini_v1(df_left, df_right)
        
        print(f"Midpoint: {mid_point}, Weighted Gini: {current_gini:.2f}")  
        
        if current_gini < best_gini:
            best_gini = current_gini
            best_split_value = mid_point
            
    return best_split_value, best_gini

In [11]:
best_split_value, best_gini = best_gini_split(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

Midpoint: 1.5, Weighted Gini: 0.36
Midpoint: 2.5, Weighted Gini: 0.27
Midpoint: 3.5, Weighted Gini: 0.15
Midpoint: 4.5, Weighted Gini: 0.00
Midpoint: 5.5, Weighted Gini: 0.13
Midpoint: 6.5, Weighted Gini: 0.22
Midpoint: 7.5, Weighted Gini: 0.29
Midpoint: 8.5, Weighted Gini: 0.33
Midpoint: 9.5, Weighted Gini: 0.37
Midpoint: 10.5, Weighted Gini: 0.40
Midpoint: 11.5, Weighted Gini: 0.42
Best split value: 4.5, Best Gini impurity: 0.00


## Optimization 1: Sliding Windows Optimization

Pseudo code:

- Sort feature
- Initialize LEFT counts to zero
- Initialize RIGHT counts to total class counts
- Move one sample at a time from RIGHT to LEFT
- Update counts
- Compute weighted Gini
- Track best boundary
- Convert best boundary to midpoint threshold

**The key advantage about this algorithm is that instead of actually splitting the feature, we calculate the total number of positive and negative label and we add 1 or subtract 1 based on the midpoint we are examining.** 

In [12]:
def gini_v2(negative: int, positive: int):
    '''
    Compute the Gini impurity for a set of labels.
    negative: refers to number of negative labels
    positive: refers to number of positive labels
    '''
    total = negative + positive
    p0 = negative / total
    p1 = positive / total

    gini = 1 - (p0**2 + p1**2)
    return gini

In [13]:
def gini_search_window(df: pd.DataFrame) -> tuple[Optional[float], float]:
    
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT Node and RIGHT Node counts
    left_total = 0
    left_positive = 0
    left_negative = 0

    right_total = n
    right_positive = (y == 1).sum()
    right_negative = (y == 0).sum()

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        left_total += 1
        right_total -= 1
        if y[i] == 1:
            left_positive += 1
            right_positive -= 1
        else:
            left_negative += 1
            right_negative -= 1
    

        # Compute weighted gini
        left_gini = gini_v2(left_positive, left_negative)
        right_gini = gini_v2(right_positive, right_negative)
        weighted_gini = ((left_total/n) * left_gini) + ((right_total/n) * right_gini)
        print(f"Weighted Gini: {weighted_gini:.2f}")

        if weighted_gini < best_gini:
            best_gini = weighted_gini
            best_split_value = (x[i] + x[i+1]) / 2
            print("Midpoint Threshold", best_split_value)
    
    return best_split_value, best_gini



In [14]:
best_split_value, best_gini = gini_search_window(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

Weighted Gini: 0.36
Midpoint Threshold 1.5
Weighted Gini: 0.27
Midpoint Threshold 2.5
Weighted Gini: 0.15
Midpoint Threshold 3.5
Weighted Gini: 0.00
Midpoint Threshold 4.5
Weighted Gini: 0.13
Weighted Gini: 0.22
Weighted Gini: 0.29
Weighted Gini: 0.33
Weighted Gini: 0.37
Weighted Gini: 0.40
Weighted Gini: 0.42
Best split value: 4.5, Best Gini impurity: 0.00


## Optimization 2: Cumulative Sum Algorithm (CUMSUM)

The sliding windows above, we still loop through all midpoint and examine the label. The next optimization

In [ ]:
def show_pos_neg_in_windows(df: pd.DataFrame):

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT Node counts
    left_total = 0
    left_positive = 0
    left_negative = 0

    right_total = n
    right_positive = (y == 1).sum()
    right_negative = (y == 0).sum()

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        left_total += 1
        right_total -= 1
        if y[i] == 1:
            left_positive += 1
            right_positive -= 1
        else:
            left_negative += 1
            right_negative -= 1
        
        print(f"Position {i}")
        print(f"Left Negative: {left_negative}")
        print(f"Left Positive: {left_positive}")
        print(f"Left Total: {left_total}")
    
        print(f"Right Negative: {right_negative}")
        print(f"Right Positive: {right_positive}")
        print(f"Right Total: {right_total}")
        print('---')


In [ ]:
show_pos_neg_in_windows(df)

In the following, we use cumulative sum for left node and compare the sliding windows.

In [ ]:
y = df['passed'].values

In [ ]:
neg_left = (y == 0).cumsum()
pos_left = (y == 1).cumsum()
print('Negative Left', neg_left)
print('Positive Left', pos_left)

In [ ]:
neg_total = neg_left[-1]
pos_total = pos_left[-1]
print('Negative Total', neg_total)
print('Positive Total', pos_total)

In [ ]:
neg_right = neg_total - neg_left
pos_right = pos_total - pos_left
print('Negative Right', neg_right)
print('Positive Right', pos_right)

In [ ]:
def gini_search_cumsum(df: pd.DataFrame) -> tuple[Optional[float], float]:
    
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT and RIGHT Node cumulative sum
    negative_left = (y == 0).cumsum()
    positive_left = (y == 1).cumsum()
    negative_total = negative_left[-1]
    positive_total = positive_left[-1]
    negative_right = negative_total - negative_left
    positive_right = positive_total - positive_left
    left_total = 0
    right_total = n

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        
        left_total += 1
        right_total -= 1
  

        # Compute weighted gini
        left_gini = gini_v2(negative_left[i], positive_left[i])
        right_gini = gini_v2(negative_right[i], positive_right[i])
        weighted_gini = ((left_total/n) * left_gini) + ((right_total/n) * right_gini)
        print(f"Weighted Gini: {weighted_gini:.2f}")

        if weighted_gini < best_gini:
            best_gini = weighted_gini
            if x[i] == x[i+1]:
                continue
            else:
                best_split_value = (x[i] + x[i+1]) / 2
            print("Midpoint Threshold", best_split_value)
        
        print('---')
    
    return best_split_value, best_gini

In [ ]:
best_split_value, best_gini = gini_search_cumsum(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

## Optimization 3: Vectorization

In [ ]:
neg_left = (y == 0).cumsum()
pos_left = (y == 1).cumsum()
print('Negative Left', neg_left)
print('Positive Left', pos_left)

In [ ]:
neg_total = neg_left[-1]
pos_total = pos_left[-1]
print('Negative Total', neg_total)
print('Positive Total', pos_total)

In [ ]:
neg_right = neg_total - neg_left
pos_right = pos_total - pos_left
print('Negative Right', neg_right)
print('Positive Right', pos_right)

In [ ]:
neg_left = neg_left[:-1]
pos_left = pos_left[:-1]
print('Negative Left', neg_left)
print('Positive Left', pos_left)

In [ ]:
neg_right = neg_right[:-1]
pos_right = pos_right[:-1]
print('Negative Right', neg_right)
print('Positive Right', pos_right)

In [ ]:
left_count = neg_left + pos_left
left_count

In [ ]:
right_count = neg_right + pos_right
right_count

In [ ]:
neg_left/left_count

In [ ]:
pos_left/left_count

In [ ]:
neg_right/right_count

In [ ]:
pos_right/right_count

In [ ]:
# Create gini for vector support
def gini_vec(negative: np.array, positive: np.array, count: np.array) -> float:
    '''
    Compute the Gini impurity for a set of labels.
    negative: refers to number of negative labels in vector
    positive: refers to number of positive labels in vector
    count: total number of labels in vector
    '''
    
    p0 = negative/count
    p1 = positive/count

    gini = 1 - (p0**2 + p1**2)
    return gini

In [ ]:
def gini_search_near_CART(df: pd.DataFrame) -> tuple[Optional[float], float]:
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Compute cumulative sum in vector form
    negative_left = (y == 0).cumsum()
    positive_left = (y == 1).cumsum()

    negative_total = negative_left[-1]
    positive_total = positive_left[-1]

    negative_right = negative_total - negative_left
    positive_right = positive_total - positive_left

    # SHIFT EVERYTHING TO SPLIT SPACE (n-1)
    negative_left = negative_left[:-1]
    positive_left = positive_left[:-1]
    negative_right = negative_right[:-1]
    positive_right = positive_right[:-1]

    count_left = negative_left + positive_left
    count_right = negative_right + positive_right

    # Step 3: Compute Weighted Gini
    left_gini = gini_vec(negative_left, positive_left, count_left)
    right_gini = gini_vec(negative_right, positive_right, count_right)
    weighted_gini = ((count_left/n) * left_gini) + ((count_right/n) * right_gini)
    print(weighted_gini)

    # Step 4: Get best gini using argmin
    best_gini_index = weighted_gini.argmin()
    best_gini = weighted_gini.min()

    best_split_value = (x[best_gini_index] + x[best_gini_index + 1]) /2
    print(best_gini, best_gini_index, best_split_value)
    
    return best_split_value, best_gini

In [ ]:
best_split_value, best_gini = gini_search_near_CART(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

## Problem with Repeated Value

In [ ]:
df2 = pd.DataFrame({'studied': [1,1,1,2,2,3,3,4,4,4,11,12], 
        'passed': [0,0,0,0,0,0,1,1,1,1,1,1]})

In [ ]:
best_split_value, best_gini = gini_search_near_CART(df2)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

### Create Decision Tree Function

In [ ]:
def gini_vec(negative: np.array, positive: np.array, count: np.array):
    '''
    Compute the Gini impurity for a set of labels.
    negative: refers to number of negative labels in vector
    positive: refers to number of positive labels in vector
    count: total number of labels in vector
    '''
    
    p0 = np.divide(negative, count, out=np.zeros_like(count, dtype=float), where=count!=0)
    p1 = np.divide(positive, count, out=np.zeros_like(count, dtype=float), where=count!=0)

    gini = 1 - (p0**2 + p1**2)
    return gini

In [ ]:
def gini_search_CART(df):
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Compute cumulative sum
    negative_left = (y == 0).cumsum()
    positive_left = (y == 1).cumsum()
    count_left = negative_left + positive_left
    # print(negative_left)
    # print(positive_left)
    negative_total = negative_left[-1]
    positive_total = positive_left[-1]
    negative_right = negative_total - negative_left
    positive_right = positive_total - positive_left
    count_right = negative_right + positive_right
    # print(negative_right)
    # print(positive_right)

    # Step 3: Compute Weighted Gini

    left_gini = gini_vec(negative_left, positive_left, count_left)
    right_gini = gini_vec(negative_right, positive_right, count_right)
    weighted_gini = ((count_left/n) * left_gini) + ((count_right/n) * right_gini)

    # Step 4: Get best gini using argmin
    best_gini_index = weighted_gini.argmin()
    best_gini = weighted_gini.min()

    best_split_value = (x[best_gini_index] + x[best_gini_index + 1]) /2
    print(best_gini, best_gini_index, best_split_value)    
    
    return best_split_value, best_gini

In [ ]:
def built_tree(df):

    if df.empty:
        return None

    # If there's only one row left, return its label
    if len(df) <= 1:
        print(f"Single row subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # If all labels are identical, return that label
    # nunique() returns the number of unique values in the 'passed' column
    if df['passed'].nunique() == 1: 
        print(f"Pure subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])


    # Find the best split based on Gini impurity
    threshold_studied, best_gini = gini_search_CART(df)
    print(f"Best split value: {threshold_studied}, Gini after split: {best_gini:.2f}")

    # Split the dataset into two subsets based on the threshold value of 'studied'
    left_subset = df[df['studied'] <= threshold_studied]
    right_subset = df[df['studied'] > threshold_studied]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node
    clf = {'threshold_studied': float(threshold_studied)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

### Build Tree 

We build a tree using current data frame.

In [ ]:
df

In [ ]:
clf = built_tree(df)

In [ ]:
clf


In [ ]:
# print tree structure in json format
print(json.dumps(clf, indent=8))


In [ ]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Threshold studied: {node['threshold_studied']}")
        print(f"{indent}├─ Left Node: <={node['threshold_studied']}")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right Node: >{node['threshold_studied']}")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent}Passed: {node}")


In [ ]:
print_tree(clf)

### Prediction / Inference

To make a prediction we pass the prediction to the tree until we reach a leaf.

In [ ]:
# make prediction function
def predict(clf, x):
    node = clf  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['threshold_studied']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [ ]:
number_of_hours_studied = 5
prediction = predict(clf, number_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: {prediction}")

## Scikit-Learn Compare

In the following, we will use Scikit-Learn to compare the tree. 

In [ ]:
from sklearn.tree import DecisionTreeClassifier

sklearn_clf = DecisionTreeClassifier(criterion='gini')

sklearn_clf.fit(df[['studied']], df['passed'])

In [ ]:
# get tree structure in json format
from sklearn.tree import export_text
tree_rules = export_text(sklearn_clf, feature_names=['studied'])
print(tree_rules)

In [ ]:
# make prediction function using sklearn
df_numbers_of_hours_studied = pd.DataFrame({'studied': [number_of_hours_studied]}) 
# need to convert to df as original data during training is in df format

sklearn_prediction = sklearn_clf.predict(df_numbers_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: {sklearn_prediction}")

## End

Here’s a clean **CART (Classification and Regression Tree) algorithm step-by-step documentation** you can reuse.

---

# CART Algorithm (Classification and Regression Tree)

## 1. Overview (intuition)

CART builds a decision tree by repeatedly asking:

> “What split best separates the data into purer groups?”

Think of it like repeatedly cutting a dataset with a knife so that each side becomes as “pure” as possible.

---

# 2. Core Idea

At each node:

* Try all possible feature splits
* Measure impurity reduction (e.g., Gini for classification, MSE for regression)
* Choose the best split
* Repeat recursively

---

# 3. CART Algorithm Steps (Classification)

## Step 1 — Start with full dataset

Given training data:

* Features: ( X )
* Labels: ( y )

This is the root node.

---

## Step 2 — For each feature, sort values

For each feature ( x_j ):

* Sort data by ( x_j )
* Consider only split points between **distinct values**

Example:

```
x: 1, 1, 2, 3, 5
```

Possible splits:

```
1.5, 2.5, 4.0
```

---

## Step 3 — Generate candidate splits

For each adjacent pair:

[
threshold = \frac{x_i + x_{i+1}}{2}
]

Only if:

[
x_i \ne x_{i+1}
]

---

## Step 4 — Split dataset

For each candidate threshold:

* Left node: ( x \le t )
* Right node: ( x > t )

---

## Step 5 — Compute impurity

For classification CART uses **Gini impurity**:

[
Gini = 1 - \sum p_k^2
]

For binary classification:

[
Gini = 1 - p_0^2 - p_1^2
]

---

## Step 6 — Compute weighted impurity

For each split:

[
G_{split} =
\frac{N_L}{N} G_L +
\frac{N_R}{N} G_R
]

Where:

* ( N_L, N_R ): number of samples in left/right node
* ( G_L, G_R ): Gini of each node

---

## Step 7 — Select best split

Choose split with:

[
\min G_{split}
]

Store:

* best feature
* best threshold
* best impurity

---

## Step 8 — Stop conditions

Stop splitting when:

* Node is pure (Gini = 0)
* Max depth reached
* Minimum samples per node reached
* No valid split improves impurity

---

## Step 9 — Split recursively

Repeat Steps 2–8 for:

* Left child node
* Right child node

---

## Step 10 — Final tree

The result is a binary tree:

* Internal nodes = decision rules
* Leaf nodes = class prediction (majority class)

---

# 4. CART Regression Variant

Same process, but replace Gini with:

## Mean Squared Error (MSE)

[
MSE = \frac{1}{n} \sum (y - \bar{y})^2
]

Split score:

[
MSE_{split} =
\frac{N_L}{N} MSE_L +
\frac{N_R}{N} MSE_R
]

---

# 5. Key Properties of CART

* Binary splits only
* Works for numerical and categorical (after encoding)
* Greedy (does NOT look ahead)
* Can overfit without pruning
* Handles non-linear decision boundaries

---

# 6. Pruning (important extension)

CART often uses **cost complexity pruning**:

[
R_\alpha(T) = R(T) + \alpha |T|
]

Where:

* ( R(T) ): training error
* ( |T| ): number of leaves
* ( \alpha ): penalty for complexity

---

# 7. Summary (short version)

At each node:

1. Sort data per feature
2. Try all valid split points
3. Compute impurity (Gini/MSE)
4. Choose best split
5. Recurse
6. Stop when stopping rules hit

---

If you want next step, I can map your **vectorized implementation line-by-line to CART steps**, which is usually where everything “clicks” for implementation understanding.
